#  BerTURK Model Eğitimi - Steam Feedback Classifier

Bu notebook, dengeli dataset (8,700 yorum) ile BerTURK modelini eğitir ve SVM/CatBoost ile karşılaştırır.

## 📊 Beklenen Sonuçlar
- **SVM Accuracy:** 92.86%
- **CatBoost Accuracy:** 94.87%
- **BerTURK Accuracy:** ~95-97% (hedef)

## 1️ Gerekli Kütüphaneleri Yükle

In [ ]:
# GPU kontrolü
import torch
print(f"GPU mevcut: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
# Gerekli kütüphaneleri yükle
!pip install transformers datasets scikit-learn pandas numpy -q

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict
import torch
from pathlib import Path
import pickle

print("✓ Kütüphaneler yüklendi!")

## 2️⃣ Veriyi Yükle ve Hazırla

In [ ]:
# Google Drive'ı bağla
from google.colab import drive
drive.mount('/content/drive')

import os

# === DOSYA YOLUNU AYARLA ===
# balanced_reviews.csv dosyasını Google Drive'ınıza yükleyin
# ve aşağıdaki yolu ona göre güncelleyin.
# Örnek: Drive'ın ana dizinine yüklediyseniz:
drive_path = '/content/drive/MyDrive/balanced_reviews.csv'

if os.path.exists(drive_path):
    print('✓ Dosya bulundu!')
else:
    print(f'❌ Dosya bulunamadı: {drive_path}')
    print('Lütfen balanced_reviews.csv dosyasını Google Drive\'ınıza yükleyin.')
    print('Sonra yukarıdaki drive_path değişkenini güncelleyin.')

# Veriyi yükle
print('\nVeri yükleniyor...')
df = pd.read_csv(drive_path)
print(f'Toplam yorum: {len(df)}')
print('\nEtiket dağılımı:')
print(df['label'].value_counts())


In [ ]:
# Etiketleri sayısal değerlere çevir
label2id = {
    'bug(hata/hile)': 0,
    'feature request(istek)': 1,
    'nötr/ çöp': 2
}

id2label = {v: k for k, v in label2id.items()}

df['labels'] = df['label'].map(label2id)

print("Etiket eşleştirmesi:")
for label, id in label2id.items():
    print(f"  {label} → {id}")

In [ ]:
# Train-test split (SVM ve CatBoost ile aynı oran)
train_df, test_df = train_test_split(
    df[['review_text', 'labels']],
    test_size=0.2,
    random_state=42,
    stratify=df['labels']
)

print(f"Train seti: {len(train_df)} örnek")
print(f"Test seti: {len(test_df)} örnek")

# Hugging Face Dataset formatına çevir
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

dataset = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print("\n✓ Dataset hazırlandı!")

## 3️⃣ BerTURK Tokenizer ve Model Yükle

In [ ]:
# BerTURK modelini yükle
model_name = "dbmdz/bert-base-turkish-cased"

print(f"BerTURK tokenizer yükleniyor: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("BerTURK model yükleniyor...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("✓ BerTURK model ve tokenizer hazır!")

In [ ]:
# Tokenization fonksiyonu
def tokenize_function(examples):
    return tokenizer(
        examples['review_text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

# Dataseti tokenize et
print("Dataset tokenize ediliyor...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=['review_text']
)

print(f"\n✓ Tokenization tamamlandı!")
print(f"Train samples: {len(tokenized_datasets['train'])}")
print(f"Test samples: {len(tokenized_datasets['test'])}")

## 4️⃣ Eğitim Parametrelerini Ayarla

In [ ]:
# Metric hesaplama fonksiyonu
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {'accuracy': accuracy}

# Training arguments
training_args = TrainingArguments(
    output_dir='./berturk_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    learning_rate=2e-5,
    fp16=torch.cuda.is_available(),  # GPU varsa mixed precision
    report_to='none',  # Weights & Biases'i kapat
)

print("✓ Training arguments hazırlandı!")
print(f"Batch size: 16 (train), 32 (eval)")
print(f"Epochs: 3")
print(f"Learning rate: 2e-5")
print(f"Mixed precision (FP16): {torch.cuda.is_available()}")

## 5️⃣ Modeli Eğit

In [ ]:
# Trainer oluştur
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
)

# Eğitimi başlat
print("="*60)
print(" BERTURK MODEL EĞİTİMİ BAŞLIYOR")
print("="*60)
print(f"Bu işlem GPU ile ~5-10 dakika sürebilir...")
print(f"CPU ile ~20-30 dakika sürebilir...")
print("="*60)

train_result = trainer.train()

print("\n" + "="*60)
print("✓ EĞİTİM TAMAMLANDI!")
print("="*60)

## 6️⃣ Modeli Değerlendir

In [ ]:
# Test seti üzerinde değerlendirme
print("Test seti üzerinde değerlendirme yapılıyor...")
eval_results = trainer.evaluate()

print("\n" + "="*60)
print("BERTURK MODEL SONUÇLARI")
print("="*60)
print(f"Accuracy: {eval_results['eval_accuracy']:.4f} (%{eval_results['eval_accuracy']*100:.2f})")
print("="*60)

# Detaylı classification report
predictions = trainer.predict(tokenized_datasets['test'])
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print("\nClassification Report:")
print(classification_report(
    y_true, 
    y_pred, 
    target_names=[id2label[i] for i in range(3)]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## 7️ Modeli Kaydet

In [ ]:
# Modeli ve tokenizer'ı kaydet
model.save_pretrained('./berturk_model')
tokenizer.save_pretrained('./berturk_model')

print("✓ BerTURK modeli kaydedildi: ./berturk_model/")

# Zip'le ve Google Drive'a kopyala
import zipfile
import os
import shutil

with zipfile.ZipFile('berturk_model.zip', 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk('./berturk_model'):
        for file in files:
            zipf.write(os.path.join(root, file))

print("✓ Model zip dosyası oluşturuldu: berturk_model.zip")

# Google Drive'a kopyala
shutil.copy('berturk_model.zip', '/content/drive/MyDrive/berturk_model.zip')
print("✓ berturk_model.zip Google Drive'a kopyalandı!")
print("  Drive'dan indirip projenize ekleyebilirsiniz.")


## 8️⃣ Model Karşılaştırması

In [ ]:
# Model karşılaştırma tablosu
import pandas as pd

comparison_df = pd.DataFrame({
    'Model': ['SVM', 'CatBoost', 'BerTURK'],
    'Accuracy': ['92.86%', '94.87%', f"{eval_results['eval_accuracy']*100:.2f}%"],
    'Eğitim Süresi': ['3.39 sn', '33.36 sn', 'GPU: ~5-10 dk'],
    'Model Tipi': ['Geleneksel ML', 'Gradient Boosting', 'Deep Learning (Transformer)'],
})

print("="*70)
print(" MODEL KARŞILAŞTIRMASI")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# En iyi modeli belirle
berturk_acc = eval_results['eval_accuracy'] * 100
if berturk_acc > 94.87:
    print(f"\n🏆 KAZANAN: BERTURK ({berturk_acc:.2f}%)")
elif berturk_acc > 92.86:
    print(f"\n🥈 İKİNCİ: BERTURK ({berturk_acc:.2f}%)")
    print(" KAZANAN: CatBoost (94.87%)")
else:
    print(f"\n🥉 ÜÇÜNCÜ: BERTURK ({berturk_acc:.2f}%)")
    print("🥇 KAZANAN: CatBoost (94.87%)")
    print(" İKİNCİ: SVM (92.86%)")

## 9️ Örnek Tahminler

In [ ]:
# Modeli test et
from transformers import pipeline

# Prediction pipeline oluştur
classifier = pipeline(
    "text-classification",
    model='./berturk_model',
    tokenizer='./berturk_model'
)

# Test örnekleri
test_reviews = [
    "Oyun sürekli çöküyor, FPS 10'a düşüyor, optimizasyon berbat",
    "Türkiye sunucusu eklerseniz çok sevinirim",
    "Güzel oyun arkadaşlarla oynayınca daha eğlenceli",
    "Hileciler yüzünden oyun oynanmıyor, anti-cheat sistemi geliştirmelisiniz",
    "Silah skinsleri için yeni bir market sistemi olmalı"
]

print("="*70)
print("🔍 BERTURK ÖRNEK TAHMİNLER")
print("="*70)
for review in test_reviews:
    result = classifier(review)[0]
    print(f"\nYorum: {review}")
    print(f"Tahmin: {result['label']} (Güven: {result['score']:.2%})")
print("="*70)